# PoofIt3D — Kaggle (T4) end-to-end run
Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add the project zip as a Kaggle Dataset (any file name). Run the cells top to bottom; the last cell writes `/kaggle/working/results.zip` (images, GLBs, `RESULTS.md`).


In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), 'Turn on a GPU accelerator'
print(torch.cuda.get_device_name(0)); subprocess.run(['nvidia-smi'])

In [ ]:
import glob, subprocess, shutil, os
# Works whether Kaggle left the upload as a zip or auto-extracted it into a folder.
hits = glob.glob('/kaggle/input/**/backend/main.py', recursive=True)
if hits:
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.rmtree('/kaggle/working/app', ignore_errors=True)
    shutil.copytree(src, '/kaggle/working/app/PoofIt3D')          # /kaggle/input is read-only
else:
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True) + glob.glob('/kaggle/working/*.zip')
    assert z, 'Upload the project zip as a Kaggle dataset first'
    subprocess.run(['unzip','-q','-o',z[0],'-d','/kaggle/working/app'], check=True)
PROJECT='/kaggle/working/app/PoofIt3D'; BACKEND=PROJECT+'/backend'
assert os.path.isfile(BACKEND+'/main.py'), 'main.py not found under '+BACKEND
print(PROJECT)

In [ ]:
import subprocess, sys
subprocess.run('apt-get update -qq && apt-get install -y -qq git ninja-build ffmpeg libgl1 libglib2.0-0', shell=True, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv','-r',BACKEND+'/requirements.txt'], check=True)

In [ ]:
import subprocess, os, shutil
from pathlib import Path
IM = Path('/kaggle/working/InstantMesh')
if not IM.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/TencentARC/InstantMesh.git',str(IM)], check=True)
ENV = Path('/kaggle/working/im-venv'); PY = ENV/'bin'/'python'; MARK = ENV/'.poofit-ok'
os.environ['UV_LINK_MODE'] = 'copy'

nvcc = shutil.which('nvcc')
CUDA_HOME = os.environ.get('CUDA_HOME') or (str(Path(nvcc).resolve().parent.parent) if nvcc else '/usr/local/cuda')
print('nvcc:', nvcc, '| CUDA_HOME:', CUDA_HOME)
subprocess.run('nvcc --version | tail -2', shell=True)
build_env = {**os.environ, 'CUDA_HOME': CUDA_HOME, 'TORCH_CUDA_ARCH_LIST': '7.5',   # 7.5 = T4
             'PATH': f"{ENV/'bin'}:{CUDA_HOME}/bin:" + os.environ['PATH']}

def pip(args, env=None):
    subprocess.run(['uv','pip','install','--python',str(PY),*args], check=True, env=env or os.environ)

if not MARK.exists():
    shutil.rmtree(ENV, ignore_errors=True)               # drop any half-built venv from a failed run
    subprocess.run(['uv','venv','--python','3.10',str(ENV)], check=True)
    pip(['--upgrade','pip','setuptools','wheel','ninja'])
    pip(['torch==2.1.0','torchvision==0.16.0','torchaudio==2.1.0','--index-url','https://download.pytorch.org/whl/cu121'])
    # torch 2.1 is built for numpy 1.x; huggingface_hub<0.26 is needed by diffusers 0.20.2.
    pip(['numpy==1.26.4','huggingface_hub==0.17.3','onnxruntime','xformers==0.0.22.post7'])
    req = [l for l in (IM/'requirements.txt').read_text().splitlines()
           if l.strip() and 'nvdiffrast' not in l and not l.lower().startswith(('huggingface','gradio'))]
    Path('/kaggle/working/im-req.txt').write_text('\n'.join(req) + '\n')
    pip(['-r','/kaggle/working/im-req.txt','numpy==1.26.4','huggingface_hub==0.17.3'])
    # nvdiffrast must be built against the already-installed torch -> no build isolation.
    pip(['--no-build-isolation','git+https://github.com/NVlabs/nvdiffrast/'], env=build_env)
    MARK.write_text('ok')

# Fail early (here, not in the 3D step): imports + JIT-compile nvdiffrast's CUDA kernels.
check = ("import torch, rembg, trimesh, numpy, diffusers, transformers;"
         "assert torch.cuda.is_available(), 'no CUDA';"
         "import nvdiffrast.torch as dr; dr.RasterizeCudaContext();"
         "print('InstantMesh env OK', torch.__version__, numpy.__version__, diffusers.__version__)")
subprocess.run([str(PY),'-c',check], cwd=IM, check=True, env=build_env)

(IM/'ckpts').mkdir(exist_ok=True)
# download with the modern hub client of the main env (the venv's old one lacks features)
from huggingface_hub import hf_hub_download
for f in ('diffusion_pytorch_model.bin','instant_mesh_large.ckpt'):
    hf_hub_download('TencentARC/InstantMesh', f, local_dir=str(IM/'ckpts'))
print('weights ready')

In [ ]:
import os, subprocess, sys, time, requests
env=os.environ.copy()
env.update(INSTANTMESH_DIR=str(IM), INSTANTMESH_PYTHON=str(PY), INSTANTMESH_CONFIG=BACKEND+'/config/instant-mesh-large-t4.yaml', DEVICE='auto')
import torch
if torch.cuda.device_count() > 1:
    env['INSTANTMESH_CUDA_VISIBLE_DEVICES'] = '1'   # 3D stage on the 2nd T4, 2D stays on the 1st
log=open('/kaggle/working/api.log','w')
subprocess.Popen([sys.executable,'-m','uvicorn','main:app','--port','8000'],cwd=BACKEND,env=env,stdout=log,stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/api/health',timeout=2).ok: break
    except Exception: time.sleep(2)
print('API up')

In [ ]:
# Full run on all 5 sketches: images + GLBs + README grids. Takes ~15-30 min on a T4.
r=subprocess.run([sys.executable,'run_e2e.py','--out','/kaggle/working/results'],cwd=BACKEND)
if r.returncode: print(open('/kaggle/working/api.log').read()[-4000:])
assert r.returncode==0

In [ ]:
subprocess.run('cd /kaggle/working && zip -qr results.zip results && ls -la results.zip', shell=True)
from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/results/*_grid.png')): display(Image(p))